This notebook demonstrates how to use the clemcore Gymnasium framework to run a
multi-player game with the game **"Taboo"**, and how to plug in a self-hosted
OpenAI-compatible model server as an agent backend.

You will learn how to:

1. Set up the environment and install dependencies.
2. Register and configure a model as an agent backend.
3. Implement a simple custom describer agent.
4. Run an automated gameplay session and inspect the interactions.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/phisad/playpen/blob/main/examples/gymnasiums/taboo.ipynb
)

# 1. Environment setup and dependency installation

## 1.1. Setup environment

We start by specifying:

- The game name (here we use `"taboo"`, but this pattern works for any 2‑player game).
- `CLEMBENCH_HOME`, the local path where the `clembench` repository is located.
  This environment variable is used by the CLI and Python APIs to locate games.

In [1]:
import os

# Specify the game name here (this code can be adapted to any 2-player game)
GAME_NAME = "taboo"

# Local clone location of the clembench repository
CLEMBENCH_HOME = os.path.expanduser("~/git/clembench")

# Expose CLEMBENCH_HOME so the clem framework can find the games
os.environ["CLEMBENCH_HOME"] = CLEMBENCH_HOME

## 1.2. Install games and dependencies

In this step we:

1. Clone the `clembench` repository (skip if you already have it).
2. Install its Python dependencies into the current Jupyter kernel.
3. Verify that `clembench` is installed and that the `taboo` game is visible.

You should:

- See a valid version printed by `clem --version`.
- See `taboo` listed by `clem list games -s taboo`.

If you do **not** see `taboo`, double‑check `CLEMBENCH_HOME` and that the clone succeeded.

In [2]:
# Clone the clembench repo (safe to re-run; git will warn if it already exists)
!git clone https://github.com/clp-research/clembench $CLEMBENCH_HOME

# Install the requirements into the Python kernel
%pip install -r $CLEMBENCH_HOME/requirements.txt

# Make tqdm usable in Jupyter notebooks
%pip install --upgrade ipywidgets jupyter_client

fatal: destination path '/Users/philippsadler/git/clembench' already exists and is not an empty directory.
  Using cached typer-0.9.4-py3-none-any.whl.metadata (14 kB)
INFO: pip is looking at multiple versions of typer-slim to determine which version is compatible with other requirements. This could take a while.
Using cached typer-0.9.4-py3-none-any.whl (45 kB)
  Attempting uninstall: typer-slim
    Found existing installation: typer-slim 0.24.0
    Uninstalling typer-slim-0.24.0:
      Successfully uninstalled typer-slim-0.24.0
  Attempting uninstall: typer
    Found existing installation: typer 0.24.0
    Uninstalling typer-0.24.0:
      Successfully uninstalled typer-0.24.0

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
  Attempting uninstall: jupyter_client
    Found existing installation: jupyter_client 8.7.0
    Uninstalling jupyter_client-8.7.0:
  

In [3]:
# Sanity check: version + confirm that the game is an available game
!clem --version
!clem list games -s $GAME_NAME

clem 3.7.0
Listing all available games (use -v option to see the whole specs)
Found '1' game specs that match the game_selector='{'game_name': 'taboo'}'
taboo:
 	Taboo game between two agents where one has to describe a word for
	the other to guess.


# 2. Register and configure a model backend

We will use a self-hosted OpenAI-compatible model server and register the model
under the name `"clp-chat"`.

This name (`"clp-chat"`) will later be used when we assign the model to a player
in the Gymnasium environment.

In [4]:
# Register the remote model under the name "clp-chat"
!clem register model -n "clp-chat" -v "backend=openai_compatible,model_id=Qwen/Qwen3-VL-30B-A3B-Instruct-FP8" --cwd

Updated model registry at /Users/philippsadler/Opts/Git/playpen/examples/gymnasium/model_registry.json successfully: {"model_name":"clp-chat","backend":"openai_compatible","lookup_source":"/Users/philippsadler/Opts/Git/playpen/examples/gymnasium/model_registry.json","model_id":"Qwen/Qwen3-VL-30B-A3B-Instruct-FP8"}


You can achieve the same registration programmatically if you prefer not to use
the CLI:

In [5]:
from clemcore.backends import ModelRegistry

registry = ModelRegistry.register("clp-chat", backend="openai_compatible", model_id="Qwen/Qwen3-VL-30B-A3B-Instruct-FP8")
registry.get_first_model_spec_that_unify_with("clp-chat")

ModelSpec({'model_name': 'clp-chat', 'backend': 'openai_compatible', 'lookup_source': '/Users/philippsadler/Opts/Git/playpen/examples/gymnasium/model_registry.json', 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Instruct-FP8'})

## 2.2. Register API key and base URL

To connect to your OpenAI-compatible model server, we also register:

- An API key
- An organisation (if applicable)
- A base URL for the endpoint

Replace the placeholders with your actual values.

In [6]:
API_KEY = "<insert-api-key-here>"
ORGANIZATION = "<insert-organisation-here>"
BASE_URL = "<insert-base-url-here>"

In [7]:
# Register the credentials and base URL for the "openai_compatible" backend
!clem register key -n "openai_compatible" -v "api_key=$API_KEY,organisation=$ORGANIZATION,base_url=$BASE_URL" --cwd

Updated key registry at /Users/philippsadler/Opts/Git/playpen/examples/gymnasium/key.json successfully: {
  "api_key": "<ins...ere>",
  "base_url": "<insert-base-url-here>",
  "organisation": "<insert-organisation-here>"
}


Programmatic equivalent:

In [8]:
from clemcore.backends import KeyRegistry
KeyRegistry.register("openai_compatible", api_key=API_KEY,organisation=ORGANIZATION, base_url=BASE_URL, force_cwd=True)

KeyRegistry(file='/Users/philippsadler/Opts/Git/playpen/examples/gymnasium/key.json', backends=[openai_compatible])

# 3. Implement a custom describer agent

Next, we implement a simple custom describer agent by subclassing `ClemAgent`.
The agent:

- Uses the `"clp-chat"` model backend.
- Generates a guess based on the full interaction history in the Taboo game.

This is just a toy example to show how you can plug in your own logic.

In [13]:
from playpen.agents import ClemAgent, ClemObservation
from clemcore.backends import load_model

class MyAgenticDescriber(ClemAgent):
    """
    Simple example agent for Taboo.

    It calls the "clp-chat" model with the current interaction history and
    uses the model's response as the next guess.
    """

    def __init__(self):
        super().__init__()
        # with a reasoning model we must allow more tokens to account for the thinking process
        self.model = load_model("clp-chat", gen_args=dict(temperature=0.7, max_tokens=None))

    def act(self, last: ClemObservation) -> str:
        # Use the full history (which usually already includes the last observation)
        _, _, response_text = self.model.generate_response(self.history)
        # Observe our own response in the interaction history
        self.observe(dict(role="user", content=response_text))
        return response_text


describer = MyAgenticDescriber()

2026-03-20 15:35:18,910 - clemcore.backends - INFO - Found registered model spec that unifies with {"model_name":"clp-chat"} -> {'model_name': 'clp-chat', 'backend': 'openai_compatible', 'lookup_source': '/Users/philippsadler/Opts/Git/playpen/examples/gymnasium/model_registry.json', 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Instruct-FP8'}
2026-03-20 15:35:18,913 - clemcore.backends - INFO - Found registry entry for backend openai_compatible -> {'backend': 'openai_compatible', 'file_name': 'openai_compatible_api.py', 'file_path': '/Users/philippsadler/Opts/Git/playpen/venv/lib/python3.10/site-packages/clemcore/backends/openai_compatible_api.py', 'lookup_source': 'packaged'}
2026-03-20 15:35:18,914 - clemcore.backends - INFO - Dynamically import backend openai_compatible
2026-03-20 15:35:18,917 - clemcore.backends - INFO - Successfully loaded clp-chat model
2026-03-20 15:35:18,917 - clemcore.backends - INFO - Loading models took: 0:00:00.003675


# 4. Run an automated gameplay session

We now:

1. Create a Gymnasium environment for the game.
2. Use our custom describer as `player_0`.
3. Use the registered `"clp-chat"` model as `player_1`.
4. Run through a single episode and log the interaction at each step.

Setting `single_pass=False` (default) allows iterating through all game instances multiple times.
You can also restrict which instances are used via the `game_instance_filter` of the `gym_env()` method.

In [14]:
from clemcore.clemgame import gym_env, episode_results_folder_callbacks
from clemcore.backends import load_model

# Create callbacks to record the interactions in a folder; here we name the folder after the models the agent uses
callbacks = episode_results_folder_callbacks(run_dir="clp-chat", result_dir_path="playpen-records", player_model_infos="MyAgenticDescriber")

game_env = gym_env(
    GAME_NAME,
    learner_agent="player_0",
    env_agents={"player_1": load_model("clp-chat", gen_args=dict(temperature=0.7, max_tokens=None))},
    callbacks=callbacks
)

2026-03-20 15:35:24,138 - clemcore.backends - INFO - Found registered model spec that unifies with {"model_name":"clp-chat"} -> {'model_name': 'clp-chat', 'backend': 'openai_compatible', 'lookup_source': '/Users/philippsadler/Opts/Git/playpen/examples/gymnasium/model_registry.json', 'model_id': 'Qwen/Qwen3-VL-30B-A3B-Instruct-FP8'}
2026-03-20 15:35:24,141 - clemcore.backends - INFO - Found registry entry for backend openai_compatible -> {'backend': 'openai_compatible', 'file_name': 'openai_compatible_api.py', 'file_path': '/Users/philippsadler/Opts/Git/playpen/venv/lib/python3.10/site-packages/clemcore/backends/openai_compatible_api.py', 'lookup_source': 'packaged'}
2026-03-20 15:35:24,143 - clemcore.backends - INFO - Dynamically import backend openai_compatible
2026-03-20 15:35:24,147 - clemcore.backends - INFO - Successfully loaded clp-chat model
2026-03-20 15:35:24,148 - clemcore.backends - INFO - Loading models took: 0:00:00.005066
2026-03-20 15:35:24,156 - clemcore.cli - INFO - Fo

In [15]:
last_obs, info = game_env.reset()
termination = False
context_response_pairs: list[tuple] = []
counter = 0
while not termination:
    print(f"Round: {counter} ...")
    action = describer(last_obs)
    obs, reward, termination, truncation, info = game_env.step(action)
    context_response_pairs.append((last_obs, action, reward))
    last_obs = obs
    counter += 1

# Reset agent state between episodes (if it maintains any internal memory)
describer.reset()

print(f"Episode took these {len(context_response_pairs)} steps:")
print("-" * 20)
for idx, (context, response, reward) in enumerate(context_response_pairs):
    print(f"Step {idx} / Reward {reward:.2f}:")
    print(f"Describer <- Context:", context)
    print(f"Describer -> Response:", response)
    print("-" * 20)

print("Final reward:", reward)

2026-03-20 15:35:28,117 - clemcore.run - INFO - Loading instance: experiment=high_en, game_id=0
Round: 0 ...
Episode took these 1 steps:
--------------------
Step 0 / Reward 0.00:
Describer <- Context: {'role': 'user', 'content': 'You are playing a collaborative word guessing game in which you have to describe a target word for another player to guess.\n\nRules:\n(a) You have to reply in the form: CLUE: <some text>. Guesses from the other player will start with GUESS.\n(b) You cannot use the target word itself, parts or morphological variants of it in your description.\n(c) In addition, the same rules apply for related words which are provided below.\n\nEnd conditions:\n(i) If you use the target word or a related word in your description, then you lose.\n(ii) If the other player can guess the target word in 3 tries, you both win.\n\nLet us start.\n\nThis is the target word that you need to describe and that the other player needs to guess:\n\nwinter\n\nRelated words are:\n\n- cold\n- s